# Chapter 11.6: Responsible AI for RecSys

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Implement content moderation filters** that remove harmful content from recommendation pipelines
2. **Detect and measure filter bubble effects** in recommendation systems
3. **Identify addictive design patterns** (infinite scroll, autoplay) and their amplification by recommenders
4. **Understand the regulatory landscape** including the EU AI Act and Digital Services Act (DSA)
5. **Build transparent recommendation explanations** that tell users why items were recommended
6. **Implement user control mechanisms** for preference management and feedback
7. **Design a content safety pipeline** for production recommendation systems

## Prerequisites

- Chapters 11.1-11.5
- Basic NLP concepts (text classification, embeddings)
- Understanding of recommendation pipelines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part11/chapter_11.6_responsible_ai.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/USERNAME/rec_system/main/notebooks/part11/chapter_11.6_responsible_ai.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import re

np.random.seed(42)
plt.style.use('seaborn-v0_8')
print("All imports successful.")

## 1. Content Moderation in Recommendations

Content moderation ensures harmful, misleading, or inappropriate content is not amplified by the recommendation system.

A multi-layer approach:
1. **Pre-indexing**: Remove clearly violating content before it enters the candidate pool
2. **Scoring-time filtering**: Apply safety classifiers during ranking
3. **Post-ranking**: Final safety check before serving

Reference: Stray et al., "What are you optimizing for? Aligning Recommender Systems with Human Values" (ICML 2020 Workshop).

> **💡 Concept:** Content safety is not just about blocking bad content. It is also about not *amplifying* borderline content. A piece of content that passes moderation should not be disproportionately promoted.

In [ ]:
# Simulate a content catalog with safety scores
rng = np.random.RandomState(42)
NUM_ITEMS = 1000
NUM_USERS = 200

# Content categories
categories = ['news', 'entertainment', 'education', 'health', 'politics', 'technology']
item_category = rng.choice(categories, size=NUM_ITEMS)

# Safety dimensions
# Score 0.0 = safe, 1.0 = most harmful
safety_scores = pd.DataFrame({
    'item_id': range(NUM_ITEMS),
    'violence': rng.beta(1, 10, NUM_ITEMS),      # mostly safe
    'misinformation': rng.beta(1.5, 8, NUM_ITEMS),  # some borderline
    'hate_speech': rng.beta(1, 15, NUM_ITEMS),    # rare but serious
    'adult_content': rng.beta(1, 12, NUM_ITEMS),
    'category': item_category,
})

# Inject some clearly harmful items
harmful_indices = rng.choice(NUM_ITEMS, size=20, replace=False)
for idx in harmful_indices[:10]:
    safety_scores.loc[idx, 'misinformation'] = rng.uniform(0.7, 1.0)
for idx in harmful_indices[10:]:
    safety_scores.loc[idx, 'hate_speech'] = rng.uniform(0.6, 1.0)

# Combined safety score: worst dimension
safety_dims = ['violence', 'misinformation', 'hate_speech', 'adult_content']
safety_scores['max_safety_risk'] = safety_scores[safety_dims].max(axis=1)

print("Safety Score Distribution (max risk per item):")
print(safety_scores['max_safety_risk'].describe().round(4))
print(f"\nItems with max_risk > 0.5: {(safety_scores['max_safety_risk'] > 0.5).sum()}")
print(f"Items with max_risk > 0.7: {(safety_scores['max_safety_risk'] > 0.7).sum()}")

In [ ]:
class ContentSafetyFilter:
    """Multi-layer content safety filter for recommendation systems."""
    
    def __init__(self, hard_threshold=0.7, soft_threshold=0.4, demotion_factor=0.5):
        self.hard_threshold = hard_threshold    # block completely
        self.soft_threshold = soft_threshold    # demote in ranking
        self.demotion_factor = demotion_factor  # multiply score by this for borderline
    
    def filter_candidates(self, item_ids, safety_df):
        """Pre-ranking: remove items above hard threshold."""
        safe_items = []
        blocked = []
        for item_id in item_ids:
            risk = safety_df.loc[item_id, 'max_safety_risk']
            if risk >= self.hard_threshold:
                blocked.append(item_id)
            else:
                safe_items.append(item_id)
        return safe_items, blocked
    
    def adjust_scores(self, item_ids, scores, safety_df):
        """Scoring-time: demote borderline content."""
        adjusted = scores.copy()
        for i, item_id in enumerate(item_ids):
            risk = safety_df.loc[item_id, 'max_safety_risk']
            if risk >= self.soft_threshold:
                adjusted[i] *= self.demotion_factor
        return adjusted
    
    def post_rank_check(self, ranked_list, safety_df, max_borderline=2):
        """Post-ranking: limit borderline content in the final list."""
        final_list = []
        borderline_count = 0
        for item_id in ranked_list:
            risk = safety_df.loc[item_id, 'max_safety_risk']
            if risk >= self.soft_threshold:
                borderline_count += 1
                if borderline_count > max_borderline:
                    continue  # skip excess borderline items
            final_list.append(item_id)
        return final_list

# Demo the safety filter
filter_ = ContentSafetyFilter(hard_threshold=0.7, soft_threshold=0.4)

# Simulate raw recommendations
raw_scores = rng.uniform(0, 1, NUM_ITEMS)
# Boost some unsafe items (simulating engagement-driven ranking)
for idx in harmful_indices:
    raw_scores[idx] *= 2.0  # harmful content often gets high engagement

all_items = list(range(NUM_ITEMS))
safe_items, blocked = filter_.filter_candidates(all_items, safety_scores)

print(f"Pre-filtering: {len(all_items)} -> {len(safe_items)} items ({len(blocked)} blocked)")

adjusted = filter_.adjust_scores(safe_items, raw_scores[safe_items], safety_scores)
ranked = [safe_items[i] for i in np.argsort(-adjusted)[:20]]
final = filter_.post_rank_check(ranked, safety_scores, max_borderline=2)

print(f"Post-ranking: top-20 -> {len(final)} items after borderline limit")

# Show risk profile of final recommendations
final_risks = safety_scores.loc[final[:10], 'max_safety_risk'].values
print(f"Risk scores of top-10 recommendations: {np.round(final_risks, 3)}")

## 2. Filter Bubbles and Echo Chambers

A filter bubble occurs when the recommendation system narrows a user's content exposure over time, reinforcing existing preferences without introducing diverse viewpoints.

$$\text{Diversity}(t+1) < \text{Diversity}(t) \quad \text{(filter bubble forming)}$$

Reference: Pariser, "The Filter Bubble: What the Internet Is Hiding from You" (2011); Nguyen et al., "Exploring the Filter Bubble" (WWW 2014).

> **⚠️ Common Pitfall:** Filter bubbles can be hard to detect because user satisfaction metrics may stay stable or even improve — users are getting more of what they like, but losing diversity of exposure.

In [ ]:
# Simulate filter bubble formation over time
NUM_STEPS = 20
NUM_SIM_USERS = 50
NUM_CATEGORIES = 6

# User starts with balanced preferences
user_prefs = np.ones((NUM_SIM_USERS, NUM_CATEGORIES)) / NUM_CATEGORIES

# Category items
items_per_category = 50

diversity_over_time = []
entropy_over_time = []

for step in range(NUM_STEPS):
    # Record current diversity (Shannon entropy of preference distribution)
    entropies = []
    for u in range(NUM_SIM_USERS):
        p = user_prefs[u] / user_prefs[u].sum()
        p = np.clip(p, 1e-10, 1.0)
        entropy = -np.sum(p * np.log2(p))
        entropies.append(entropy)
    entropy_over_time.append(np.mean(entropies))
    
    # Recommend based on user preferences (reinforcement)
    for u in range(NUM_SIM_USERS):
        # Recommend from top preference categories
        rec_probs = user_prefs[u] ** 2  # squared = more focused on top prefs
        rec_probs /= rec_probs.sum()
        recommended_categories = rng.choice(NUM_CATEGORIES, size=10, p=rec_probs)
        
        # User clicks (biased towards recommended categories)
        for cat in recommended_categories:
            if rng.random() < 0.3:  # 30% click rate
                user_prefs[u, cat] += 0.1  # reinforce preference

# Compare with a diversity-aware system
user_prefs_div = np.ones((NUM_SIM_USERS, NUM_CATEGORIES)) / NUM_CATEGORIES
entropy_over_time_div = []

for step in range(NUM_STEPS):
    entropies = []
    for u in range(NUM_SIM_USERS):
        p = user_prefs_div[u] / user_prefs_div[u].sum()
        p = np.clip(p, 1e-10, 1.0)
        entropies.append(-np.sum(p * np.log2(p)))
    entropy_over_time_div.append(np.mean(entropies))
    
    for u in range(NUM_SIM_USERS):
        # Diversity-aware: mix preference-based and exploratory
        rec_probs = 0.7 * (user_prefs_div[u] / user_prefs_div[u].sum()) + 0.3 / NUM_CATEGORIES
        rec_probs /= rec_probs.sum()
        recommended_categories = rng.choice(NUM_CATEGORIES, size=10, p=rec_probs)
        
        for cat in recommended_categories:
            if rng.random() < 0.3:
                user_prefs_div[u, cat] += 0.1

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(NUM_STEPS), entropy_over_time, 'o-', label='Standard RecSys', color='coral', linewidth=2)
ax.plot(range(NUM_STEPS), entropy_over_time_div, 's-', label='Diversity-Aware RecSys', color='green', linewidth=2)
ax.set_xlabel('Time Step')
ax.set_ylabel('Average User Preference Entropy (bits)')
ax.set_title('Filter Bubble Formation: Entropy of User Preferences Over Time')
ax.legend()
ax.set_ylim(0, np.log2(NUM_CATEGORIES) + 0.1)
ax.axhline(y=np.log2(NUM_CATEGORIES), color='grey', linestyle=':', label='Maximum entropy')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Addictive Patterns and Engagement Maximisation

Recommendation systems can amplify addictive patterns:

| Pattern | Mechanism | Risk |
|---------|-----------|------|
| Infinite scroll | No natural stopping point | Excessive time spent |
| Autoplay | Removes user agency | Passive consumption |
| Notification-driven re-engagement | Creates urgency | Compulsive checking |
| Intermittent rewards | Variable-ratio reinforcement | Habit formation |

> **💡 Concept:** The tension between engagement maximisation and user wellbeing is one of the most important challenges in responsible AI for recommendation systems.

In [ ]:
# Simulate the effect of engagement-maximising vs wellbeing-aware recommendations
n_sim_users = 100
n_days = 60

# Engagement-maximising: session duration increases but satisfaction decreases
engagement_sessions = np.zeros((n_sim_users, n_days))
engagement_satisfaction = np.zeros((n_sim_users, n_days))
wellbeing_sessions = np.zeros((n_sim_users, n_days))
wellbeing_satisfaction = np.zeros((n_sim_users, n_days))

for u in range(n_sim_users):
    base_session = rng.uniform(20, 60)  # minutes
    base_satisfaction = rng.uniform(0.6, 0.9)
    
    for d in range(n_days):
        # Engagement-max: session grows, satisfaction declines
        engagement_sessions[u, d] = base_session * (1 + 0.02 * d) + rng.normal(0, 5)
        engagement_satisfaction[u, d] = max(0, base_satisfaction * (1 - 0.005 * d) + rng.normal(0, 0.05))
        
        # Wellbeing-aware: session stable, satisfaction stable
        wellbeing_sessions[u, d] = base_session * (1 + 0.005 * d) + rng.normal(0, 5)
        wellbeing_satisfaction[u, d] = max(0, base_satisfaction * (1 - 0.001 * d) + rng.normal(0, 0.05))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(np.mean(engagement_sessions, axis=0), label='Engagement-Max', color='red', linewidth=2)
axes[0].plot(np.mean(wellbeing_sessions, axis=0), label='Wellbeing-Aware', color='green', linewidth=2)
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Avg Session Duration (min)')
axes[0].set_title('Session Duration Over Time')
axes[0].legend()

axes[1].plot(np.mean(engagement_satisfaction, axis=0), label='Engagement-Max', color='red', linewidth=2)
axes[1].plot(np.mean(wellbeing_satisfaction, axis=0), label='Wellbeing-Aware', color='green', linewidth=2)
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Avg User Satisfaction')
axes[1].set_title('User Satisfaction Over Time')
axes[1].legend()

plt.suptitle('Engagement Maximisation vs User Wellbeing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Regulatory Landscape

### EU AI Act (2024)
- Recommender systems are classified as **limited risk** or **high risk** depending on context
- Transparency requirements: users must know they are interacting with AI
- Algorithmic auditing requirements for high-risk systems

### Digital Services Act (DSA, 2022)
- Very Large Online Platforms (VLOPs) must provide transparency on recommendation algorithms
- Users must have access to a non-profiling-based recommendation option
- Annual risk assessments for systemic risks (election integrity, public health, etc.)

> **🔑 Pro Tip:** Designing for compliance from the start is much cheaper than retrofitting. Build in transparency, user control, and audit logging from day one.

In [ ]:
# Compliance checklist implementation
class ComplianceChecker:
    """Check recommendation system against regulatory requirements."""
    
    def __init__(self):
        self.checks = [
            ('transparency_disclosure', 'Users informed that recommendations are AI-generated'),
            ('non_profiling_option', 'Non-personalised recommendation option available'),
            ('user_data_access', 'Users can access their recommendation profile data'),
            ('opt_out_mechanism', 'Users can opt out of personalised recommendations'),
            ('content_moderation', 'Content safety filters are in place'),
            ('audit_logging', 'Recommendation decisions are logged for audit'),
            ('bias_assessment', 'Regular bias and fairness assessments conducted'),
            ('risk_assessment', 'Annual systemic risk assessment (VLOPs)'),
        ]
    
    def run_checks(self, system_config):
        """Run compliance checks against system configuration."""
        results = []
        for check_id, description in self.checks:
            status = system_config.get(check_id, False)
            results.append({
                'Check': check_id,
                'Description': description,
                'Status': 'PASS' if status else 'FAIL',
            })
        return pd.DataFrame(results)

# Example system configuration
system_config = {
    'transparency_disclosure': True,
    'non_profiling_option': True,
    'user_data_access': True,
    'opt_out_mechanism': True,
    'content_moderation': True,
    'audit_logging': False,  # Not yet implemented
    'bias_assessment': True,
    'risk_assessment': False,  # Not yet done
}

checker = ComplianceChecker()
report = checker.run_checks(system_config)
print("=== Regulatory Compliance Report ===")
print(report.to_string(index=False))
print(f"\nPassed: {(report['Status'] == 'PASS').sum()}/{len(report)}")

## 5. Transparency: Recommendation Explanations

Users should understand *why* an item was recommended. Common explanation types:
- "Because you watched X" (content-based)
- "Users like you also enjoyed..." (collaborative)
- "Trending in your area" (popularity/context)
- "Based on your interest in [topic]" (topic-based)

In [ ]:
class RecommendationExplainer:
    """Generate human-readable explanations for recommendations."""
    
    def __init__(self, user_history, item_features, item_names):
        self.user_history = user_history  # dict: user_id -> list of item_ids
        self.item_features = item_features  # dict: item_id -> feature dict
        self.item_names = item_names  # dict: item_id -> name
    
    def explain(self, user_id, recommended_item_id, method='content'):
        """Generate explanation for why an item was recommended."""
        item_name = self.item_names.get(recommended_item_id, f'Item {recommended_item_id}')
        item_feat = self.item_features.get(recommended_item_id, {})
        
        if method == 'content':
            # Find most similar item in user history
            history = self.user_history.get(user_id, [])
            if history:
                similar_item = history[-1]  # simplified: use last watched
                similar_name = self.item_names.get(similar_item, f'Item {similar_item}')
                return f"Recommended '{item_name}' because you enjoyed '{similar_name}'"
            return f"Recommended '{item_name}' based on your interests"
        
        elif method == 'topic':
            category = item_feat.get('category', 'general')
            return f"Recommended '{item_name}' based on your interest in {category}"
        
        elif method == 'popularity':
            return f"'{item_name}' is trending right now"
        
        return f"'{item_name}' was recommended for you"

# Demo
item_names = {i: f"Article_{i}" for i in range(NUM_ITEMS)}
item_feats = {i: {'category': item_category[i]} for i in range(NUM_ITEMS)}
user_hist = {u: list(rng.choice(NUM_ITEMS, size=5, replace=False)) for u in range(10)}

explainer = RecommendationExplainer(user_hist, item_feats, item_names)

print("=== Recommendation Explanations ===")
for u in range(3):
    rec_item = rng.choice(NUM_ITEMS)
    for method in ['content', 'topic', 'popularity']:
        explanation = explainer.explain(u, rec_item, method=method)
        print(f"  User {u}, {method}: {explanation}")
    print()

## 6. User Control Mechanisms

Giving users control over their recommendation experience improves trust and satisfaction.

| Control | Implementation |
|---------|---------------|
| "Not interested" | Remove item/topic from future recommendations |
| Topic preferences | Let users boost/mute specific categories |
| History management | Allow users to delete watch/click history |
| Diversity slider | Let users control explore/exploit balance |

In [ ]:
class UserControlledRecommender:
    """Recommendation system with user control mechanisms."""
    
    def __init__(self, model_scores, item_categories, num_items):
        self.model_scores = model_scores.copy()
        self.item_categories = item_categories
        self.num_items = num_items
        self.blocked_items = defaultdict(set)       # user -> set of blocked items
        self.blocked_categories = defaultdict(set)   # user -> set of muted categories
        self.boosted_categories = defaultdict(set)   # user -> set of boosted categories
        self.diversity_preference = defaultdict(lambda: 0.5)  # 0=focused, 1=diverse
    
    def block_item(self, user_id, item_id):
        self.blocked_items[user_id].add(item_id)
    
    def mute_category(self, user_id, category):
        self.blocked_categories[user_id].add(category)
    
    def boost_category(self, user_id, category):
        self.boosted_categories[user_id].add(category)
    
    def set_diversity(self, user_id, level):
        self.diversity_preference[user_id] = np.clip(level, 0, 1)
    
    def recommend(self, user_id, k=10):
        scores = self.model_scores[user_id].copy()
        
        # Apply blocks
        for item in self.blocked_items[user_id]:
            scores[item] = -np.inf
        
        # Apply category muting and boosting
        for item in range(self.num_items):
            cat = self.item_categories[item]
            if cat in self.blocked_categories[user_id]:
                scores[item] = -np.inf
            if cat in self.boosted_categories[user_id]:
                scores[item] *= 1.5
        
        # Apply diversity: add noise proportional to diversity preference
        div_level = self.diversity_preference[user_id]
        noise = rng.randn(self.num_items) * div_level * 0.5
        scores += noise
        
        # Return top-k
        valid_mask = scores > -np.inf
        valid_items = np.where(valid_mask)[0]
        valid_scores = scores[valid_mask]
        top_k_idx = np.argsort(-valid_scores)[:k]
        return valid_items[top_k_idx].tolist()

# Demo user controls
model_scores_sim = rng.uniform(0, 1, size=(NUM_USERS, NUM_ITEMS))
rec_system = UserControlledRecommender(model_scores_sim, item_category, NUM_ITEMS)

user = 0
print("Default recommendations (top 5):")
default_recs = rec_system.recommend(user, k=5)
for item in default_recs:
    print(f"  Item {item} ({item_category[item]})")

# User mutes 'politics' and boosts 'technology'
rec_system.mute_category(user, 'politics')
rec_system.boost_category(user, 'technology')
rec_system.set_diversity(user, 0.8)

print("\nAfter user controls (mute politics, boost tech, high diversity):")
controlled_recs = rec_system.recommend(user, k=5)
for item in controlled_recs:
    print(f"  Item {item} ({item_category[item]})")

## 7. Exercises

### Exercise 1: Build a Content Safety Pipeline

In [ ]:
# 🏋️ Exercise 1: Content Safety Pipeline
#
# TODO: Extend the ContentSafetyFilter to:
# 1. Track blocked items and reasons (which safety dimension triggered the block)
# 2. Compute safety metrics: % of served items that are borderline,
#    category-level block rates
# 3. Implement a "circuit breaker": if >10% of a user's recommendations
#    are borderline, switch to a safe fallback (e.g., popular + safe items)
# 4. Visualise the safety pipeline's impact on recommendation quality (NDCG)

# --- YOUR CODE HERE ---

# --- END YOUR CODE ---

### Exercise 2: Filter Bubble Detection

In [ ]:
# 🏋️ Exercise 2: Filter Bubble Detection and Mitigation
#
# TODO:
# 1. Implement a filter bubble detector that monitors:
#    a. Category entropy of recommendations over time
#    b. Intra-list diversity trend
#    c. User exploration rate (% of new categories clicked)
# 2. When a filter bubble is detected (entropy drops below threshold),
#    inject diverse "exploration" items into the recommendation list
# 3. Simulate 50 time steps and show that the mitigation prevents
#    the entropy from dropping

# --- YOUR CODE HERE ---

# --- END YOUR CODE ---

### Exercise 3: Recommendation Explanation Quality

In [ ]:
# 🏋️ Exercise 3: Explanation Quality Metrics
#
# TODO: Implement metrics to evaluate explanation quality:
# 1. Fidelity: does the explanation reflect the actual model's reasoning?
#    (Measure: if we remove the explained feature, does the score drop?)
# 2. Diversity: are explanations varied or repetitive?
# 3. Specificity: does the explanation distinguish this recommendation
#    from alternatives?
#
# Compute these metrics for a set of 100 recommendations.

# --- YOUR CODE HERE ---

# --- END YOUR CODE ---

## Summary

In this notebook we covered:

- **Content moderation**: multi-layer safety filtering (pre-indexing, scoring-time, post-ranking)
- **Filter bubbles**: how recommendations narrow content exposure and how to detect/mitigate
- **Addictive patterns**: the tension between engagement maximisation and user wellbeing
- **Regulatory landscape**: EU AI Act and DSA requirements for recommendation systems
- **Transparency**: generating meaningful explanations for recommendations
- **User control**: giving users agency over their recommendation experience

**Key takeaway:** Responsible AI in recommendation systems requires a holistic approach that considers safety, transparency, user agency, and compliance. Building these capabilities from the start is essential, not optional.